# Notebook 03 v2 — NSL-KDD calibration (hybrid Platt/isotonic)

**What this notebook does:**

Fits per-class calibrators on the calibration set carved out in Notebook 01 v2 (M6 fix), applies them to the test set, and reports per-class calibration quality.

**Calibration strategy (decided in response to TA's rare-class caveat comment):**

- **Isotonic regression** for classes with n_calib ≥ 30 (parameter-rich, flexible monotonic fit)
- **Platt scaling** (sigmoid, 2 parameters) for classes with n_calib < 30 (parameter-efficient)

NSL calibration set sizes:
- Normal: 13,469 → isotonic
- DoS: 9,186 → isotonic
- Probe: 2,331 → isotonic
- R2L: 199 → isotonic
- **U2R: 10 → Platt** (the only rare class needing Platt)

**Per-class metrics reported:**

- **Brier score**: lower is better, measures probabilistic accuracy
- **ECE (Expected Calibration Error)**: lower is better, measures how well predicted probabilities match observed frequencies
- **Strategy used**: explicit per-class recording of isotonic vs Platt

**M6 fix:** Calibrators fit on `calib_proba.npy` (from training-data-derived calibration set), applied to `test_proba.npy` (official untouched test set).

## 1. Setup

In [10]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

!git pull
print(f'\n✓ Ready in: {os.getcwd()}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already up to date.

✓ Ready in: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from collections import Counter

from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

SEED = 42
np.random.seed(SEED)

DATASET = 'nsl_kdd_v2'
PROCESSED = Path(REPO) / 'data' / 'processed' / DATASET
MODELS_DIR = Path(REPO) / 'models' / DATASET
PREDS_DIR = MODELS_DIR / 'predictions'
CALIB_OUT_DIR = Path(REPO) / 'calibrators' / DATASET
CALIB_OUT_DIR.mkdir(parents=True, exist_ok=True)

# Decision threshold for Platt vs isotonic
PLATT_THRESHOLD = 30

# Number of bins for ECE computation
ECE_N_BINS = 10

print(f'Dataset: {DATASET}')
print(f'Platt threshold: n_calib < {PLATT_THRESHOLD} → Platt; n_calib ≥ {PLATT_THRESHOLD} → isotonic')
print(f'Output dir: {CALIB_OUT_DIR}')

Dataset: nsl_kdd_v2
Platt threshold: n_calib < 30 → Platt; n_calib ≥ 30 → isotonic
Output dir: /content/drive/MyDrive/XIDS_Research/xids-research/calibrators/nsl_kdd_v2


## 2. Load calibration set labels and class mappings

In [3]:
# Labels for the calibration set (used to fit calibrators)
y_calib_b = np.load(PROCESSED / 'y_calib_binary.npy')
y_calib_5 = np.load(PROCESSED / 'y_calib_5class.npy')

# Labels for the test set (used to evaluate calibrated probabilities)
y_test_b = np.load(PROCESSED / 'y_test_binary.npy')
y_test_5 = np.load(PROCESSED / 'y_test_5class.npy')

with open(PROCESSED / 'class_mappings.json') as f:
    class_info = json.load(f)
INT_TO_CATEGORY = {int(k): v for k, v in class_info['multiclass_5'].items()}
CLASS_NAMES_5 = [INT_TO_CATEGORY[i] for i in range(5)]

print(f'Calibration set: {len(y_calib_b):,} samples')
print(f'Test set: {len(y_test_b):,} samples')
print()

# Per-class calibration sizes
calib_counts_5 = Counter(y_calib_5)
calib_counts_b = Counter(y_calib_b)

print('Per-class calibration set sizes:')
print(f'  Binary:')
for c in [0, 1]:
    n = calib_counts_b[c]
    strat = 'Platt' if n < PLATT_THRESHOLD else 'isotonic'
    print(f'    {"Normal" if c == 0 else "Attack":8s}: n={n:>6,}  → {strat}')

print(f'  5-class:')
for c in range(5):
    n = calib_counts_5[c]
    strat = 'Platt' if n < PLATT_THRESHOLD else 'isotonic'
    print(f'    {CLASS_NAMES_5[c]:8s}: n={n:>6,}  → {strat}')

Calibration set: 25,195 samples
Test set: 22,544 samples

Per-class calibration set sizes:
  Binary:
    Normal  : n=13,469  → isotonic
    Attack  : n=11,726  → isotonic
  5-class:
    Normal  : n=13,469  → isotonic
    DoS     : n= 9,186  → isotonic
    Probe   : n= 2,331  → isotonic
    R2L     : n=   199  → isotonic
    U2R     : n=    10  → Platt


## 3. Helper functions

In [4]:
def fit_calibrator(p_calib, y_indicator, n_class):
    """Fit either isotonic or Platt depending on calibration sample size.

    Args:
        p_calib: raw probabilities for class c on calibration set, shape (n_calib,)
        y_indicator: 1[y_calib == c] for class c, shape (n_calib,)
        n_class: number of calibration samples in class c

    Returns:
        (calibrator, strategy_name)
    """
    if n_class >= PLATT_THRESHOLD:
        # Isotonic regression: monotonic stepwise fit
        cal = IsotonicRegression(out_of_bounds='clip')
        cal.fit(p_calib, y_indicator)
        return cal, 'isotonic'
    else:
        # Platt scaling: logistic regression on the single feature p_calib
        cal = LogisticRegression(C=1e10, solver='lbfgs')  # ~unregularised sigmoid
        cal.fit(p_calib.reshape(-1, 1), y_indicator)
        return cal, 'platt'

def apply_calibrator(calibrator, strategy, p_test):
    """Apply fitted calibrator to test probabilities."""
    if strategy == 'isotonic':
        return calibrator.predict(p_test)
    else:  # platt
        return calibrator.predict_proba(p_test.reshape(-1, 1))[:, 1]

def expected_calibration_error(probs, labels, n_bins=10):
    """Compute ECE: weighted average gap between accuracy and confidence per bin.

    For multiclass, this is the standard formulation using predicted-class confidence.
    For per-class one-vs-rest, probs and labels are binary (P[y=c], 1[y=c]).
    """
    bin_edges = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    n = len(probs)
    for i in range(n_bins):
        lo, hi = bin_edges[i], bin_edges[i + 1]
        if i == n_bins - 1:
            mask = (probs >= lo) & (probs <= hi)
        else:
            mask = (probs >= lo) & (probs < hi)
        if mask.sum() == 0:
            continue
        bin_conf = probs[mask].mean()
        bin_acc = labels[mask].mean()
        weight = mask.sum() / n
        ece += weight * abs(bin_conf - bin_acc)
    return float(ece)

def calibrate_model_per_class(p_calib_2d, y_calib, p_test_2d, y_test, n_classes):
    """Apply per-class calibration with hybrid Platt/isotonic.

    Args:
        p_calib_2d: raw probs on calib set, shape (n_calib, n_classes)
        y_calib: integer labels for calib set, shape (n_calib,)
        p_test_2d: raw probs on test set, shape (n_test, n_classes)
        y_test: integer labels for test set, shape (n_test,)
        n_classes: number of classes

    Returns:
        dict with calibrated probs, per-class strategies, per-class Brier and ECE
    """
    p_test_cal = np.zeros_like(p_test_2d)
    strategies = {}
    brier_pre = {}
    brier_post = {}
    ece_pre = {}
    ece_post = {}

    calib_counts = Counter(y_calib)

    for c in range(n_classes):
        # One-vs-rest indicator vectors
        y_calib_c = (y_calib == c).astype(int)
        y_test_c = (y_test == c).astype(int)
        p_calib_c = p_calib_2d[:, c]
        p_test_c = p_test_2d[:, c]

        n_c = calib_counts[c]
        cal, strat = fit_calibrator(p_calib_c, y_calib_c, n_c)
        p_test_cal[:, c] = apply_calibrator(cal, strat, p_test_c)
        strategies[c] = strat

        # Per-class Brier (pre-cal and post-cal)
        brier_pre[c]  = float(brier_score_loss(y_test_c, p_test_c))
        brier_post[c] = float(brier_score_loss(y_test_c, p_test_cal[:, c]))

        # Per-class ECE
        ece_pre[c]  = expected_calibration_error(p_test_c,  y_test_c, ECE_N_BINS)
        ece_post[c] = expected_calibration_error(p_test_cal[:, c], y_test_c, ECE_N_BINS)

    # Renormalise so each row sums to 1 (calibrated probs may not sum to 1 naturally)
    row_sums = p_test_cal.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums == 0, 1, row_sums)  # avoid div by zero
    p_test_cal_renorm = p_test_cal / row_sums

    return {
        'p_test_cal_raw': p_test_cal,           # before renormalisation
        'p_test_cal': p_test_cal_renorm,        # after renormalisation
        'strategies': strategies,
        'brier_pre': brier_pre,
        'brier_post': brier_post,
        'ece_pre': ece_pre,
        'ece_post': ece_post,
    }

def calibrate_binary(p_calib_2d, y_calib, p_test_2d, y_test):
    """For binary models: calibrate only P[class=1] (attack probability).

    Binary 5-class doesn't apply here; this is for the binary-task models specifically.
    """
    p_calib_attack = p_calib_2d[:, 1]
    p_test_attack = p_test_2d[:, 1]
    y_calib_attack = y_calib  # already 0/1
    y_test_attack = y_test

    calib_counts = Counter(y_calib)
    n_attack = calib_counts[1]

    cal, strat = fit_calibrator(p_calib_attack, y_calib_attack, n_attack)
    p_test_cal_attack = apply_calibrator(cal, strat, p_test_attack)
    p_test_cal_attack = np.clip(p_test_cal_attack, 0, 1)

    # 2-column form for consistency: [P[normal], P[attack]]
    p_test_cal_2d = np.column_stack([1 - p_test_cal_attack, p_test_cal_attack])

    return {
        'p_test_cal': p_test_cal_2d,
        'strategies': {1: strat},  # only the attack class was calibrated
        'brier_pre':  {1: float(brier_score_loss(y_test_attack, p_test_attack))},
        'brier_post': {1: float(brier_score_loss(y_test_attack, p_test_cal_attack))},
        'ece_pre':  {1: expected_calibration_error(p_test_attack, y_test_attack, ECE_N_BINS)},
        'ece_post': {1: expected_calibration_error(p_test_cal_attack, y_test_attack, ECE_N_BINS)},
    }

print('✓ Helper functions loaded')

✓ Helper functions loaded


## 4. Calibrate all 9 models

In [5]:
ALL_MODELS = [
    ('rf_binary_cw',     'binary'),
    ('xgb_binary_cw',    'binary'),
    ('dnn_binary_cw',    'binary'),
    ('rf_5class_smote',  '5class'),
    ('xgb_5class_smote', '5class'),
    ('dnn_5class_smote', '5class'),
    ('rf_5class_cw',     '5class'),
    ('xgb_5class_cw',    '5class'),
    ('dnn_5class_cw',    '5class'),
]

calibration_summary = {}

for name, task in ALL_MODELS:
    print(f'\nCalibrating {name} ({task})...')

    p_calib = np.load(PREDS_DIR / f'{name}_calib_proba.npy')
    p_test = np.load(PREDS_DIR / f'{name}_test_proba.npy')

    # Ensure 2D shape
    if p_calib.ndim == 1:
        p_calib = np.column_stack([1 - p_calib, p_calib])
    if p_test.ndim == 1:
        p_test = np.column_stack([1 - p_test, p_test])

    if task == 'binary':
        result = calibrate_binary(p_calib, y_calib_b, p_test, y_test_b)
    else:  # 5class
        result = calibrate_model_per_class(p_calib, y_calib_5, p_test, y_test_5, n_classes=5)

    # Save calibrated probabilities
    np.save(CALIB_OUT_DIR / f'{name}_test_proba_calibrated.npy', result['p_test_cal'])

    # Report per-class summary
    print(f'  Strategies: {result["strategies"]}')
    if task == '5class':
        for c in range(5):
            cname = CLASS_NAMES_5[c]
            print(f'  {cname:8s}: Brier {result["brier_pre"][c]:.4f} → {result["brier_post"][c]:.4f}'
                  f'   ECE {result["ece_pre"][c]:.4f} → {result["ece_post"][c]:.4f}'
                  f'   [{result["strategies"][c]}]')
    else:
        print(f'  Attack  : Brier {result["brier_pre"][1]:.4f} → {result["brier_post"][1]:.4f}'
              f'   ECE {result["ece_pre"][1]:.4f} → {result["ece_post"][1]:.4f}'
              f'   [{result["strategies"][1]}]')

    calibration_summary[name] = {
        'task': task,
        'strategies': {int(k): v for k, v in result['strategies'].items()},
        'brier_pre':  {int(k): v for k, v in result['brier_pre'].items()},
        'brier_post': {int(k): v for k, v in result['brier_post'].items()},
        'ece_pre':  {int(k): v for k, v in result['ece_pre'].items()},
        'ece_post': {int(k): v for k, v in result['ece_post'].items()},
    }

# Save summary
with open(CALIB_OUT_DIR / 'calibration_summary.json', 'w') as f:
    json.dump(calibration_summary, f, indent=2)

print(f'\n✓ All 9 models calibrated. Summary saved to {CALIB_OUT_DIR}/calibration_summary.json')


Calibrating rf_binary_cw (binary)...
  Strategies: {1: 'isotonic'}
  Attack  : Brier 0.1633 → 0.1675   ECE 0.1900 → 0.1868   [isotonic]

Calibrating xgb_binary_cw (binary)...
  Strategies: {1: 'isotonic'}
  Attack  : Brier 0.1988 → 0.1941   ECE 0.2078 → 0.2065   [isotonic]

Calibrating dnn_binary_cw (binary)...
  Strategies: {1: 'isotonic'}
  Attack  : Brier 0.1789 → 0.1805   ECE 0.1803 → 0.1828   [isotonic]

Calibrating rf_5class_smote (5class)...
  Strategies: {0: 'isotonic', 1: 'isotonic', 2: 'isotonic', 3: 'isotonic', 4: 'platt'}
  Normal  : Brier 0.1509 → 0.1915   ECE 0.1810 → 0.2107   [isotonic]
  DoS     : Brier 0.0679 → 0.0609   ECE 0.0674 → 0.0634   [isotonic]
  Probe   : Brier 0.0361 → 0.0415   ECE 0.0227 → 0.0395   [isotonic]
  R2L     : Brier 0.1045 → 0.1193   ECE 0.1089 → 0.1220   [isotonic]
  U2R     : Brier 0.0021 → 0.0029   ECE 0.0028 → 0.0029   [platt]

Calibrating xgb_5class_smote (5class)...
  Strategies: {0: 'isotonic', 1: 'isotonic', 2: 'isotonic', 3: 'isotonic', 

## 5. Summary table

In [6]:
# Build a paper-ready summary table
rows = []
for name, task in ALL_MODELS:
    s = calibration_summary[name]
    if task == '5class':
        # Macro-average Brier and ECE across the 5 classes
        brier_pre_macro = np.mean(list(s['brier_pre'].values()))
        brier_post_macro = np.mean(list(s['brier_post'].values()))
        ece_pre_macro = np.mean(list(s['ece_pre'].values()))
        ece_post_macro = np.mean(list(s['ece_post'].values()))
        platt_classes = [CLASS_NAMES_5[c] for c, st in s['strategies'].items() if st == 'platt']
        rows.append({
            'Model': name,
            'Task': '5-class',
            'Brier pre (macro)':  round(brier_pre_macro,  5),
            'Brier post (macro)': round(brier_post_macro, 5),
            'ECE pre (macro)':    round(ece_pre_macro,    5),
            'ECE post (macro)':   round(ece_post_macro,   5),
            'Platt for': ','.join(platt_classes) if platt_classes else '—',
        })
    else:
        rows.append({
            'Model': name,
            'Task': 'binary',
            'Brier pre (macro)':  round(s['brier_pre'][1],  5),
            'Brier post (macro)': round(s['brier_post'][1], 5),
            'ECE pre (macro)':    round(s['ece_pre'][1],    5),
            'ECE post (macro)':   round(s['ece_post'][1],   5),
            'Platt for': '—' if s['strategies'][1] == 'isotonic' else 'attack class',
        })

df = pd.DataFrame(rows)
print('\nNSL v2 — Hybrid Platt/Isotonic Calibration Results')
print('=' * 110)
print(df.to_string(index=False))
print('=' * 110)

# Save as paper-ready CSV
TABLES_DIR = Path(REPO) / 'results' / 'tables'
TABLES_DIR.mkdir(parents=True, exist_ok=True)
csv_path = TABLES_DIR / 'nslkdd_v2_calibration_summary.csv'
df.to_csv(csv_path, index=False)
print(f'\nSaved: {csv_path}')


NSL v2 — Hybrid Platt/Isotonic Calibration Results
           Model    Task  Brier pre (macro)  Brier post (macro)  ECE pre (macro)  ECE post (macro) Platt for
    rf_binary_cw  binary            0.16328             0.16747          0.19004           0.18681         —
   xgb_binary_cw  binary            0.19880             0.19408          0.20782           0.20652         —
   dnn_binary_cw  binary            0.17887             0.18054          0.18034           0.18284         —
 rf_5class_smote 5-class            0.07231             0.08323          0.07658           0.08771       U2R
xgb_5class_smote 5-class            0.07953             0.08039          0.08225           0.08275       U2R
dnn_5class_smote 5-class            0.08437             0.08734          0.08360           0.08761       U2R
    rf_5class_cw 5-class            0.07914             0.08641          0.08200           0.08999       U2R
   xgb_5class_cw 5-class            0.08033             0.08012          0.0

## 6. Per-class breakdown (5-class models only)

In [7]:
# Detailed per-class table for the 5-class models
perclass_rows = []
for name, task in ALL_MODELS:
    if task != '5class':
        continue
    s = calibration_summary[name]
    for c in range(5):
        perclass_rows.append({
            'Model': name,
            'Class': CLASS_NAMES_5[c],
            'n_calib': calib_counts_5[c],
            'Strategy': s['strategies'][c],
            'Brier pre':  round(s['brier_pre'][c],  5),
            'Brier post': round(s['brier_post'][c], 5),
            'ECE pre':    round(s['ece_pre'][c],    5),
            'ECE post':   round(s['ece_post'][c],   5),
        })

df_perclass = pd.DataFrame(perclass_rows)
print('\nNSL v2 — Per-class calibration breakdown')
print('=' * 110)
print(df_perclass.to_string(index=False))
print('=' * 110)

csv_path = TABLES_DIR / 'nslkdd_v2_calibration_perclass.csv'
df_perclass.to_csv(csv_path, index=False)
print(f'\nSaved: {csv_path}')


NSL v2 — Per-class calibration breakdown
           Model  Class  n_calib Strategy  Brier pre  Brier post  ECE pre  ECE post
 rf_5class_smote Normal    13469 isotonic    0.15093     0.19154  0.18104   0.21071
 rf_5class_smote    DoS     9186 isotonic    0.06785     0.06095  0.06743   0.06341
 rf_5class_smote  Probe     2331 isotonic    0.03612     0.04148  0.02267   0.03948
 rf_5class_smote    R2L      199 isotonic    0.10448     0.11932  0.10893   0.12202
 rf_5class_smote    U2R       10    platt    0.00214     0.00289  0.00285   0.00293
xgb_5class_smote Normal    13469 isotonic    0.18238     0.18405  0.19426   0.19821
xgb_5class_smote    DoS     9186 isotonic    0.06288     0.06199  0.06516   0.06305
xgb_5class_smote  Probe     2331 isotonic    0.04124     0.03969  0.03761   0.03398
xgb_5class_smote    R2L      199 isotonic    0.10892     0.11368  0.11188   0.11590
xgb_5class_smote    U2R       10    platt    0.00225     0.00256  0.00232   0.00261
dnn_5class_smote Normal    13469 i

## 7. Sanity checks

In [8]:
# Verify that calibration didn't break anything
print('Sanity checks on calibrated probabilities:')
print('-' * 70)

for name, task in ALL_MODELS:
    p_cal = np.load(CALIB_OUT_DIR / f'{name}_test_proba_calibrated.npy')

    # Check 1: shape
    if task == 'binary':
        expected_cols = 2
    else:
        expected_cols = 5
    shape_ok = p_cal.shape[1] == expected_cols

    # Check 2: range [0, 1]
    range_ok = (p_cal >= 0).all() and (p_cal <= 1).all()

    # Check 3: rows sum to ~1 (within 0.001)
    row_sums = p_cal.sum(axis=1)
    sum_ok = np.allclose(row_sums, 1, atol=0.001)
    max_dev = float(np.abs(row_sums - 1).max())

    # Check 4: no NaN or inf
    finite_ok = np.isfinite(p_cal).all()

    all_ok = shape_ok and range_ok and sum_ok and finite_ok
    flag = '✓' if all_ok else '✗'
    print(f'  {flag} {name:<22} shape={p_cal.shape}  range_ok={range_ok}  '
          f'sum_max_dev={max_dev:.5f}  finite={finite_ok}')

print('\nAll calibrated outputs validated.')

Sanity checks on calibrated probabilities:
----------------------------------------------------------------------
  ✓ rf_binary_cw           shape=(22544, 2)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ xgb_binary_cw          shape=(22544, 2)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ dnn_binary_cw          shape=(22544, 2)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ rf_5class_smote        shape=(22544, 5)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ xgb_5class_smote       shape=(22544, 5)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ dnn_5class_smote       shape=(22544, 5)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ rf_5class_cw           shape=(22544, 5)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ xgb_5class_cw          shape=(22544, 5)  range_ok=True  sum_max_dev=0.00000  finite=True
  ✓ dnn_5class_cw          shape=(22544, 5)  range_ok=True  sum_max_dev=0.00000  finite=True

All calibrated outputs validated.


## 8. Commit

In [9]:
os.chdir(REPO)
!git add notebooks/03_nsl_calibration_v2.ipynb
!git add calibrators/nsl_kdd_v2/
!git add results/tables/nslkdd_v2_calibration_summary.csv
!git add results/tables/nslkdd_v2_calibration_perclass.csv
!git status --short
!git commit -m 'Notebook 03-NSL v2: hybrid Platt/isotonic calibration with per-class Brier and ECE'
!git push origin main

Refresh index: 100% (239/239), done.
A  calibrators/nsl_kdd_v2/calibration_summary.json
 M notebooks/01_cic_data_exploration_v2.ipynb
 M notebooks/01_data_exploration_v2.ipynb
 M notebooks/01_unsw_data_exploration_v2.ipynb
 M notebooks/02_cic_train_models_v2.ipynb
 M notebooks/02_nsl_rf_retrain_v2.ipynb
 M notebooks/02_train_models_v2.ipynb
 M notebooks/02_unsw_train_models_v2.ipynb
AM notebooks/03_nsl_calibration_v2.ipynb
A  results/tables/nslkdd_v2_calibration_perclass.csv
A  results/tables/nslkdd_v2_calibration_summary.csv
?? calibrators/nsl_kdd/
?? calibrators/unsw_nb15/
?? docs/v2_rebuild_progress_day1_day2_v7.md
?? models/
?? notebooks/02b_unsw_dnn_diagnostic.ipynb
?? results/figures/unsw_dnn_5class_diagnostic.png
?? shap_values/unsw_nb15/
[main bb7cdd3] Notebook 03-NSL v2: hybrid Platt/isotonic calibration with per-class Brier and ECE
 4 files changed, 869 insertions(+)
 create mode 100644 calibrators/nsl_kdd_v2/calibration_summary.json
 create mode 100644 notebooks/03_nsl_calib

In [11]:
import numpy as np
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score

REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
PREDS_DIR = Path(REPO) / 'models' / 'nsl_kdd_v2' / 'predictions'
HYBRID_DIR = Path(REPO) / 'calibrators' / 'nsl_kdd_v2'
PROCESSED = Path(REPO) / 'data' / 'processed' / 'nsl_kdd_v2'

y_test = np.load(PROCESSED / 'y_test_5class.npy')

models_5class = ['rf_5class_smote', 'xgb_5class_smote', 'dnn_5class_smote',
                 'rf_5class_cw', 'xgb_5class_cw', 'dnn_5class_cw']

print('NSL HYBRID Platt/Isotonic — argmax preservation check')
print(f'{"Model":<22} {"% flipped":>10} {"Acc pre":>9} {"Acc post":>10} {"Δ acc":>9} {"F1m pre":>9} {"F1m post":>10} {"Δ F1m":>9}')
print('-' * 95)
for name in models_5class:
    p_pre = np.load(PREDS_DIR / f'{name}_test_proba.npy')
    p_post = np.load(HYBRID_DIR / f'{name}_test_proba_calibrated.npy')
    pred_pre = p_pre.argmax(axis=1)
    pred_post = p_post.argmax(axis=1)
    n_flipped = (pred_pre != pred_post).sum()
    pct_flipped = 100 * n_flipped / len(pred_pre)

    acc_pre = accuracy_score(y_test, pred_pre)
    acc_post = accuracy_score(y_test, pred_post)
    f1_pre = f1_score(y_test, pred_pre, average='macro', zero_division=0)
    f1_post = f1_score(y_test, pred_post, average='macro', zero_division=0)

    print(f'{name:<22} {pct_flipped:>9.2f}% {acc_pre:>9.4f} {acc_post:>10.4f} {acc_post-acc_pre:>+9.4f} '
          f'{f1_pre:>9.4f} {f1_post:>10.4f} {f1_post-f1_pre:>+9.4f}')

NSL HYBRID Platt/Isotonic — argmax preservation check
Model                   % flipped   Acc pre   Acc post     Δ acc   F1m pre   F1m post     Δ F1m
-----------------------------------------------------------------------------------------------
rf_5class_smote             3.61%    0.7474     0.7746   +0.0271    0.5165     0.5415   +0.0250
xgb_5class_smote            0.21%    0.7869     0.7858   -0.0011    0.6138     0.5935   -0.0203
dnn_5class_smote            2.72%    0.7812     0.7698   -0.0114    0.5747     0.5600   -0.0147
rf_5class_cw                4.74%    0.7338     0.7759   +0.0421    0.4683     0.5328   +0.0645
xgb_5class_cw               0.26%    0.7842     0.7859   +0.0017    0.5948     0.5793   -0.0154
dnn_5class_cw               7.35%    0.7716     0.7745   +0.0028    0.5471     0.5678   +0.0207


In [12]:
import numpy as np
from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score

REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
PREDS_DIR = Path(REPO) / 'models' / 'nsl_kdd_v2' / 'predictions'
DIRICHLET_DIR = Path(REPO) / 'calibrators' / 'nsl_kdd_v2_dirichlet'
PROCESSED = Path(REPO) / 'data' / 'processed' / 'nsl_kdd_v2'

y_test = np.load(PROCESSED / 'y_test_5class.npy')

models_5class = ['rf_5class_smote', 'xgb_5class_smote', 'dnn_5class_smote',
                 'rf_5class_cw', 'xgb_5class_cw', 'dnn_5class_cw']

print('NSL DIRICHLET ODIR — argmax preservation check')
print(f'{"Model":<22} {"% flipped":>10} {"Acc pre":>9} {"Acc post":>10} {"Δ acc":>9} {"F1m pre":>9} {"F1m post":>10} {"Δ F1m":>9}')
print('-' * 95)
for name in models_5class:
    p_pre = np.load(PREDS_DIR / f'{name}_test_proba.npy')
    p_post = np.load(DIRICHLET_DIR / f'{name}_test_proba_calibrated.npy')
    pred_pre = p_pre.argmax(axis=1)
    pred_post = p_post.argmax(axis=1)
    n_flipped = (pred_pre != pred_post).sum()
    pct_flipped = 100 * n_flipped / len(pred_pre)

    acc_pre = accuracy_score(y_test, pred_pre)
    acc_post = accuracy_score(y_test, pred_post)
    f1_pre = f1_score(y_test, pred_pre, average='macro', zero_division=0)
    f1_post = f1_score(y_test, pred_post, average='macro', zero_division=0)

    print(f'{name:<22} {pct_flipped:>9.2f}% {acc_pre:>9.4f} {acc_post:>10.4f} {acc_post-acc_pre:>+9.4f} '
          f'{f1_pre:>9.4f} {f1_post:>10.4f} {f1_post-f1_pre:>+9.4f}')

NSL DIRICHLET ODIR — argmax preservation check
Model                   % flipped   Acc pre   Acc post     Δ acc   F1m pre   F1m post     Δ F1m
-----------------------------------------------------------------------------------------------
rf_5class_smote             1.74%    0.7474     0.7347   -0.0127    0.5165     0.4654   -0.0511
xgb_5class_smote            1.13%    0.7869     0.7779   -0.0090    0.6138     0.5287   -0.0851
dnn_5class_smote            2.58%    0.7812     0.7676   -0.0136    0.5747     0.4965   -0.0781
rf_5class_cw                0.75%    0.7338     0.7294   -0.0044    0.4683     0.4575   -0.0108
xgb_5class_cw               1.08%    0.7842     0.7755   -0.0087    0.5948     0.5218   -0.0730
dnn_5class_cw               5.61%    0.7716     0.7734   +0.0018    0.5471     0.5218   -0.0253


In [13]:
import json
import numpy as np
from pathlib import Path

REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'

print('=' * 100)
print('RECONCILIATION REPORT — loaded from CSV/JSON on disk, authoritative source')
print('=' * 100)

# UNSW dnn_5class_cw — resolve the F1m inconsistency
print('\n## UNSW Dirichlet dnn_5class_cw inspection:')
unsw_dir_summary = Path(REPO) / 'calibrators' / 'unsw_nb15_v2_dirichlet' / 'calibration_summary.json'
with open(unsw_dir_summary) as f:
    d = json.load(f)
print(f"  Method: {d['dnn_5class_cw'].get('method', 'n/a')}")
print(f"  Brier pre: {d['dnn_5class_cw']['brier_pre']}")
print(f"  Brier post: {d['dnn_5class_cw']['brier_post']}")
print(f"  Macro Brier pre: {np.mean(list(d['dnn_5class_cw']['brier_pre'].values())):.5f}")
print(f"  Macro Brier post: {np.mean(list(d['dnn_5class_cw']['brier_post'].values())):.5f}")

# Compute actual F1m post-Dirichlet on UNSW dnn_5class_cw
import sklearn.metrics as skm
PROCESSED = Path(REPO) / 'data' / 'processed' / 'unsw_nb15_v2'
PROBS_DIR = Path(REPO) / 'models' / 'unsw_nb15_v2' / 'probabilities'
DIR_DIR = Path(REPO) / 'calibrators' / 'unsw_nb15_v2_dirichlet'

y_test = np.load(PROCESSED / 'y_test_5class.npy')
p_pre = np.load(PROBS_DIR / 'dnn_5class_cw_test_proba.npy')
p_post = np.load(DIR_DIR / 'dnn_5class_cw_test_proba_calibrated.npy')

acc_pre = skm.accuracy_score(y_test, p_pre.argmax(axis=1))
acc_post = skm.accuracy_score(y_test, p_post.argmax(axis=1))
f1_pre = skm.f1_score(y_test, p_pre.argmax(axis=1), average='macro', zero_division=0)
f1_post = skm.f1_score(y_test, p_post.argmax(axis=1), average='macro', zero_division=0)

print(f"\n  AUTHORITATIVE numbers (from disk):")
print(f"  Acc pre = {acc_pre:.4f}, Acc post = {acc_post:.4f}, Δ acc = {acc_post-acc_pre:+.4f}")
print(f"  F1m pre = {f1_pre:.4f}, F1m post = {f1_post:.4f}, Δ F1m = {f1_post-f1_pre:+.4f}")

# UNSW three-way macro comparison
print('\n## UNSW three-way macro Brier comparison:')
unsw_hybrid = Path(REPO) / 'calibrators' / 'unsw_nb15_v2' / 'calibration_summary.json'
with open(unsw_hybrid) as f:
    h = json.load(f)

models = ['rf_binary_cw', 'xgb_binary_cw', 'dnn_binary_cw',
          'rf_5class_smote', 'xgb_5class_smote', 'dnn_5class_smote',
          'rf_5class_cw', 'xgb_5class_cw', 'dnn_5class_cw']

print(f'  {"Model":<20} {"Brier pre":>10} {"Brier hybrid":>13} {"Brier dirichlet":>16} {"Winner":>10}')
print('  ' + '-' * 80)

for m in models:
    b_pre = np.mean(list(d[m]['brier_pre'].values()))
    b_dir = np.mean(list(d[m]['brier_post'].values()))
    h_post = h[m]['brier_post']
    b_hyb = np.mean([v for k, v in h_post.items()])

    methods = {'pre': b_pre, 'hybrid': b_hyb, 'dirichlet': b_dir}
    winner = min(methods, key=methods.get)
    print(f'  {m:<20} {b_pre:>10.5f} {b_hyb:>13.5f} {b_dir:>16.5f} {winner:>10}')

print('\n## NSL three-way macro Brier comparison:')
nsl_hybrid = Path(REPO) / 'calibrators' / 'nsl_kdd_v2' / 'calibration_summary.json'
nsl_dir = Path(REPO) / 'calibrators' / 'nsl_kdd_v2_dirichlet' / 'calibration_summary.json'
with open(nsl_hybrid) as f:
    nh = json.load(f)
with open(nsl_dir) as f:
    nd = json.load(f)

print(f'  {"Model":<20} {"Brier pre":>10} {"Brier hybrid":>13} {"Brier dirichlet":>16} {"Winner":>10}')
print('  ' + '-' * 80)
winner_counts = {'pre': 0, 'hybrid': 0, 'dirichlet': 0}
for m in models:
    b_pre = np.mean(list(nd[m]['brier_pre'].values()))
    b_dir = np.mean(list(nd[m]['brier_post'].values()))
    h_post = nh[m]['brier_post']
    b_hyb = np.mean([v for k, v in h_post.items()])

    methods = {'pre': b_pre, 'hybrid': b_hyb, 'dirichlet': b_dir}
    winner = min(methods, key=methods.get)
    winner_counts[winner] += 1
    print(f'  {m:<20} {b_pre:>10.5f} {b_hyb:>13.5f} {b_dir:>16.5f} {winner:>10}')

print(f'\n  NSL winner counts: {winner_counts}')

RECONCILIATION REPORT — loaded from CSV/JSON on disk, authoritative source

## UNSW Dirichlet dnn_5class_cw inspection:
  Method: dirichlet_odir
  Brier pre: {'0': 0.14460700537465393, '1': 0.06111635220624702, '2': 0.04936428244073529, '3': 0.17259947082067176, '4': 0.037904647301788066}
  Brier post: {'0': 0.11837549775668672, '1': 0.04643929687416541, '2': 0.033420176757069556, '3': 0.14777949594269876, '4': 0.006602176786789271}
  Macro Brier pre: 0.09312
  Macro Brier post: 0.07052

  AUTHORITATIVE numbers (from disk):
  Acc pre = 0.6017, Acc post = 0.7078, Δ acc = +0.1061
  F1m pre = 0.4774, F1m post = 0.4447, Δ F1m = -0.0327

## UNSW three-way macro Brier comparison:
  Model                 Brier pre  Brier hybrid  Brier dirichlet     Winner
  --------------------------------------------------------------------------------
  rf_binary_cw            0.09506       0.10611          0.10345        pre
  xgb_binary_cw           0.09669       0.10875          0.10608        pre
  dnn_

In [ ]:
import os
os.chdir('/content/drive/MyDrive/XIDS_Research/xids-research')
# (Save the downloaded file as docs/calibration_findings_exact.md, overwriting v1)
!git add docs/calibration_findings_exact.md
!git commit -m "Calibration findings doc v2: reconciled numbers from disk, all TBDs resolved"
!git push origin main


In [14]:
import os
os.chdir('/content/drive/MyDrive/XIDS_Research/xids-research')
# Save downloaded files to docs/
!git add docs/calibration_findings_exact_v3.md
!git add docs/calibration_session_memory_v3.md
!git commit -m "Calibration documents v3: audit pass, corrected count error, expanded tables"
!git push origin main


fatal: pathspec 'docs/calibration_findings_exact_v3.md' did not match any files
fatal: pathspec 'docs/calibration_session_memory_v3.md' did not match any files
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   notebooks/03_nsl_calibration_v2.ipynb
	modified:   notebooks/03c_cic_calibration_dirichlet_v2.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	docs/calibration_findings_exact_v2.md
	docs/calibration_findings_exact_v3.md
	docs/calibration_session_memory.md
	docs/calibration_session_memory_v3.md
	docs/v2_rebuild_progress_day1_day2_v7.md
	models/
	shap_values/unsw_nb15/

no changes added to commit (use "git add" and/or "git commit -a")
Everything up-to-date
